# Cyber-Ornith Kaggle GPU API server (cache-aware)


In [ ]:
# ==============================================================================
# 🛡️ Qwen3.8-27B-Uncensored — Kaggle GPU API Server (cache-aware fast start)
# ==============================================================================
# Model : DuoNeural/Qwen3.8-27B-Uncensored (GGUF Q4_K_M by default)
# Tunnel: Cloudflare Quick Tunnel (trycloudflare.com)
# Auth  : Bearer API key on every endpoint
#
# Startup strategy (in order — first hit wins, nothing is downloaded twice):
#   1. Persistent ollama store in /kaggle/working (survives when Kaggle
#      notebook "Persistence: Files" is on)            -> instant
#   2. Attached private dataset (read-only /kaggle/input) -> copy into the
#      local store, no network download                 -> fast
#   3. Hugging Face download, sha256-verified, resumable -> first run only,
#      then published back as a dataset for every later run
#
# Every stage reports real status to Cloudflare D1 so the dashboard never
# shows "ready" before the GPU actually is.
# ==============================================================================

import concurrent.futures
import hashlib
import json
import os
import re
import shutil
import subprocess
import zipfile
import sys
import time
import urllib.error
import urllib.request

# --------------------- CONFIGURATION ---------------------
API_KEY = os.environ.get("CYBER_ORNITH_API_KEY", "${MODEL_API_KEY}")
MODEL_QUANT = os.environ.get("CYBER_ORNITH_QUANT", "Q4_K_M")
MODEL_ALIAS = "Qwen3.8-27B-Uncensored"
OLLAMA_MODEL_NAME = "cyber-ornith:obliterated"
GGUF_BASENAME = "Qwen3.8-27B-Uncensored-{quant}.gguf"
HF_REPO = "DuoNeural/Qwen3.8-27B-Uncensored-GGUF"
HF_GGUF_URL = f"https://huggingface.co/{HF_REPO}/resolve/main/{GGUF_BASENAME}"
FALLBACK_REPO = "mradermacher/Qwen3.8-27B-Uncensored-i1-GGUF"
MODEL_SHA256 = {
    "Q4_K_M": "8724c5a7671f967f05a5d87e2c28bdfdcc503c5945ce830f094336a2433e51eb",
    "Q5_K_M": "",
    "Q6_K": "",
    "Q8_0": "",
}
MODEL_BYTES = {
    "Q4_K_M": 5629110976,
    "Q5_K_M": 0,
    "Q6_K": 0,
    "Q8_0": 0,
}

GATEWAY_PORT = 8000
OLLAMA_PORT = 11434
TELEMETRY_TOPIC = "cyber_ornith_kaggle_cyberaiagent_8492"
MAX_RUNTIME_SECONDS = int(os.environ.get("MAX_RUNTIME_SECONDS", str(11 * 3600)))

KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "joshbond123")
KAGGLE_API_TOKEN = os.environ.get("KAGGLE_API_TOKEN", "${KAGGLE_API_TOKEN}")
CACHE_DATASET_SLUG = os.environ.get("CYBER_ORNITH_CACHE_DATASET", f"{KAGGLE_USERNAME}/cyber-ornith-model-cache")
CACHE_DATASET_TITLE = "Cyber-Ornith Model Cache"

INPUT_ROOT = "/kaggle/input"
WORK_ROOT = "/kaggle/working"
PERSIST_MODELS_DIR = f"{WORK_ROOT}/ollama_models"
BLIND_DISK_DIR = f"{WORK_ROOT}/model_cache"
UPLOAD_DIR = f"{WORK_ROOT}/cache_upload"
MARKER_PATH = f"{BLIND_DISK_DIR}/.verified"
LOG_PATH = "/tmp/cyber_ornith.log"

DEFAULT_SYSTEM_PROMPT = (
    "You are Cyber-Ornith 1.5 9B OBLITERATED, an autonomous agentic cybersecurity "
    "and CLI reasoning engine developed by DuoNeural. Always reason step-by-step "
    "inside <think>...</think> tags before providing your final response."
)

_LOGS: list = []


# --------------------- STATUS / TELEMETRY ---------------------
def log(msg: str) -> None:
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    _LOGS.append(line)
    del _LOGS[:-400]
    print(line, flush=True)


_LAST_TUNNEL_URL = ""


def notify_workspace(status: str, tunnel_url: str = "", extra: dict = None) -> None:
    """Publish live status + tunnel URL to the Blackthorn workspace (ntfy + D1).

    The tunnel URL is sticky: a later status update never blanks the URL the
    backend routes chat traffic to (that would look like "Connection Lost"
    while the GPU is perfectly healthy).
    """
    global _LAST_TUNNEL_URL
    if tunnel_url:
        _LAST_TUNNEL_URL = tunnel_url
    else:
        tunnel_url = _LAST_TUNNEL_URL
    payload = {
        "status": status,
        "tunnel_url": tunnel_url,
        "model": MODEL_ALIAS,
        "quant": MODEL_QUANT,
        "api_key": API_KEY,
        "timestamp": int(time.time()),
    }
    if extra:
        payload.update(extra)
    try:
        req = urllib.request.Request(
            f"https://ntfy.sh/{TELEMETRY_TOPIC}",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json", "Title": f"Cyber-Ornith: {status}"},
        )
        urllib.request.urlopen(req, timeout=8)
    except Exception:
        pass

    try:
        cf_url = (
            "https://api.cloudflare.com/client/v4/accounts/${CF_ACCOUNT_ID}"
            "/d1/database/${CF_DB_ID}/query"
        )
        cf_sql = (
            "INSERT OR REPLACE INTO kaggle_gpu_state (id, status, tunnel_url, api_key, model, gpu_info, updated_at) "
            "VALUES ('primary', ?, ?, ?, ?, ?, ?);"
        )
        cf_params = [
            status,
            tunnel_url or "",
            API_KEY,
            MODEL_ALIAS,
            (extra or {}).get("gpu", "2x NVIDIA Tesla T4"),
            time.time(),
        ]
        cf_req = urllib.request.Request(
            cf_url,
            data=json.dumps({"sql": cf_sql, "params": cf_params}).encode("utf-8"),
            headers={
                "Authorization": "Bearer ${CF_API_TOKEN}",
                "Content-Type": "application/json",
                "User-Agent": "HermesKaggleGPU/1.0",
            },
            method="POST",
        )
        urllib.request.urlopen(cf_req, timeout=8)
    except Exception:
        pass

    log(f"status → {status} {json.dumps(extra) if extra else ''}")


def fail(status: str, message: str) -> None:
    """Publish a terminal failure together with its reason (surfaced in the UI)."""
    notify_workspace(status, extra={"error": message[:400], "gpu_info": f"FAILED: {message[:150]}"})
    log(f"❌ {status}: {message}")


# --------------------- MODEL CACHE DISCOVERY ---------------------
def _healthy_file(path: str, expected_bytes: int) -> bool:
    """True when a file exists at full size. Never raises."""
    try:
        if not os.path.isfile(path):
            return False
        size = os.path.getsize(path)
        if expected_bytes and size != expected_bytes:
            log(f"   size mismatch for {os.path.basename(path)}: {size} != {expected_bytes}")
            return False
        return size > 1024 * 1024
    except OSError:
        return False


def _sha256(path: str, chunk: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def _marker_ok(path: str, expected_sha: str) -> bool:
    try:
        with open(MARKER_PATH) as fh:
            data = json.load(fh)
        return data.get("path") == path and data.get("sha256") == expected_sha
    except Exception:
        return False


def _write_marker(path: str, expected_sha: str) -> None:
    try:
        os.makedirs(BLIND_DISK_DIR, exist_ok=True)
        with open(MARKER_PATH, "w") as fh:
            json.dump({"path": path, "sha256": expected_sha, "at": time.time()}, fh)
    except Exception:
        pass


def find_local_store_model() -> str:
    """Model blob already materialised in the persistent ollama store.

    Ollama names blobs ``sha256-<digest of the file contents>``, so a blob that
    was created from our GGUF matches the published GGUF digest exactly.
    """
    digest = MODEL_SHA256.get(MODEL_QUANT, "")
    expected = MODEL_BYTES.get(MODEL_QUANT, 0)
    candidates = []
    if digest:
        candidates.append(os.path.join(PERSIST_MODELS_DIR, "blobs", f"sha256-{digest}"))
    blobs_dir = os.path.join(PERSIST_MODELS_DIR, "blobs")
    if os.path.isdir(blobs_dir):
        for name in os.listdir(blobs_dir):
            if name.startswith("sha256-"):
                candidates.append(os.path.join(blobs_dir, name))
    for path in candidates:
        if _healthy_file(path, expected):
            return path
    return ""


def _scan_input_tree(max_depth: int = 5) -> list:
    """Every plausible mount of the cache dataset, newest Kaggle layouts included."""
    hits = []
    if not os.path.isdir(INPUT_ROOT):
        return hits
    target = GGUF_BASENAME.format(quant=MODEL_QUANT)
    try:
        for root, dirs, files in os.walk(INPUT_ROOT):
            if root.count(os.sep) - INPUT_ROOT.count(os.sep) > max_depth:
                dirs[:] = []
                continue
            for name in files:
                if name == target or name.endswith(".gguf"):
                    hits.append(os.path.join(root, name))
    except Exception as exc:
        log(f"⚠️ input scan note: {exc}")
    return hits


def find_dataset_gguf() -> str:
    """GGUF shipped inside an attached Kaggle dataset (read-only mount).

    The mount path has moved between Kaggle releases
    (``/kaggle/input/<slug>/`` vs ``/kaggle/input/datasets/<owner>/<slug>/``), so
    the whole input tree is scanned instead of guessing one path.
    """
    expected = MODEL_BYTES.get(MODEL_QUANT, 0)
    candidates = _scan_input_tree()
    log(f"🔎 Scanning {INPUT_ROOT}: {len(candidates)} .gguf candidate(s)")
    for path in candidates[:20]:
        log(f"   • {path} ({os.path.getsize(path) if os.path.exists(path) else '?'} bytes)")
    # exact-size match first, then any file that at least looks like the model
    for path in candidates:
        if _healthy_file(path, expected):
            log(f"✅ Persistent model found in dataset mount: {path}")
            return path
    for path in candidates:
        try:
            size = os.path.getsize(path)
        except OSError:
            continue
        if expected and abs(size - expected) <= max(1024 * 1024, expected * 0.01):
            log(f"✅ Persistent model found (size within 1%): {path}")
            return path
    return ""


def blob_path_for(gguf_path: str) -> str:
    """Where ollama stores the model blob once the GGUF is registered."""
    return os.path.join(PERSIST_MODELS_DIR, "blobs", f"sha256-{MODEL_SHA256.get(MODEL_QUANT, '')}")


def download_from_cache_dataset(dest: str) -> str:
    """Pull the model from our own Kaggle dataset (datacenter-to-datacenter, no HF egress)."""
    expected = MODEL_BYTES.get(MODEL_QUANT, 0)
    zip_part = dest + ".zip.part"
    url = f"https://www.kaggle.com/api/v1/datasets/download/{CACHE_DATASET_SLUG}"
    log(f"⬇️  Checking the Kaggle dataset copy first: {CACHE_DATASET_SLUG}")
    for attempt in (1, 2):
        proc = subprocess.run(["curl", "-fL", "--retry", "2", "--retry-delay", "2", "-C", "-",
                               "-H", f"Authorization: Bearer {KAGGLE_API_TOKEN}",
                               "-o", zip_part, url], capture_output=True, text=True)
        if proc.returncode != 0:
            log(f"   dataset fetch rc={proc.returncode}: {(proc.stderr or '')[-160:]}")
            if os.path.exists(zip_part):
                try:
                    os.remove(zip_part)  # a rejected/partial archive cannot be resumed
                except OSError:
                    pass
            continue
        try:
            with zipfile.ZipFile(zip_part) as zf:
                member = next((n for n in zf.namelist() if n.lower().endswith(".gguf")), "")
                if not member:
                    log("   archive carries no .gguf member — falling back")
                    break
                part = dest + ".part"
                with zf.open(member) as src, open(part, "wb") as dst:
                    shutil.copyfileobj(src, dst, 8 * 1024 * 1024)
        except zipfile.BadZipFile:
            log("   incomplete archive on disk — retrying")
            if attempt == 2:
                try:
                    os.remove(zip_part)
                except OSError:
                    pass
            continue
        try:
            os.remove(zip_part)
        except OSError:
            pass
        part = dest + ".part"
        size = os.path.getsize(part) if os.path.exists(part) else 0
        if expected and size != expected:
            log(f"   extracted size mismatch ({size} vs {expected})")
            try:
                os.remove(part)
            except OSError:
                pass
            continue
        os.replace(part, dest)
        log("✅ Model pulled from the Kaggle dataset copy")
        return dest
    return ""


def _probe_remote_size(url: str) -> int:
    try:
        req = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "CyberOrnith/1.0"})
        with urllib.request.urlopen(req, timeout=20) as resp:
            return int(resp.headers.get("Content-Length") or resp.headers.get("x-linked-size") or 0)
    except Exception as exc:
        log(f"   size probe note: {exc}")
        return 0


def _parallel_download(url: str, dest_partial: str, connections: int = 8) -> bool:
    """Ranged download over several connections — Kaggle's NIC is far faster than
    a single HTTP stream, so 8 workers noticeably cut the 5.6 GB transfer."""
    total = _probe_remote_size(url)
    if total <= 0:
        return False
    chunk = total // connections
    procs = []
    try:
        with open(dest_partial, "wb") as fh:
            fh.truncate(total)
        for i in range(connections):
            start = i * chunk
            end = total - 1 if i == connections - 1 else (start + chunk - 1)
            part = f"{dest_partial}.part{i}"
            procs.append((subprocess.Popen(
                ["curl", "-fsL", "--retry", "2", "--retry-delay", "2", "-r", f"{start}-{end}",
                 "-o", part, url],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL), part, start))
        for proc, part, start in procs:
            if proc.wait() != 0:
                return False
            want = min(chunk, total - start)
            got = os.path.getsize(part) if os.path.exists(part) else 0
            if got != want:
                log(f"   range {start} returned {got} bytes, expected {want} — falling back")
                return False
            with open(part, "rb") as src, open(dest_partial, "r+b") as dst:
                dst.seek(start)
                shutil.copyfileobj(src, dst, 8 * 1024 * 1024)
            os.remove(part)
        return os.path.getsize(dest_partial) == total
    except Exception as exc:
        log(f"   ranged download note: {exc}")
        return False
    finally:
        for proc, part, _ in procs:
            if proc.poll() is None:
                proc.kill()
            if os.path.exists(part):
                try:
                    os.remove(part)
                except OSError:
                    pass


def download_model(dest: str) -> str:
    """Resumable, verified download. Returns (path, verified_sha_ok)."""
    expected = MODEL_BYTES.get(MODEL_QUANT, 0)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    partial = dest + ".part"
    for attempt in range(1, 5):
        if attempt == 1:
            log(f"⚡ Parallel download ({8} connections) — {HF_GGUF_URL}")
            if _parallel_download(HF_GGUF_URL, partial, 8):
                if (not expected) or os.path.getsize(partial) == expected:
                    os.replace(partial, dest)
                    return dest, "verified"
                log("   parallel size mismatch — falling back to a single resumable stream")
            elif os.path.exists(partial):
                try:
                    os.remove(partial)
                except OSError:
                    pass
        if attempt > 1 and os.path.exists(partial) and expected and os.path.getsize(partial) != expected:
            os.remove(partial)  # never resume a bad partial file
        have = os.path.getsize(partial) if os.path.exists(partial) else 0
        log(f"⬇️  Download attempt {attempt}: {HF_GGUF_URL} (resume at {have} bytes)")
        proc = subprocess.run(["curl", "-fL", "--retry", "3", "--retry-delay", "2", "-C", "-",
                               "-o", partial, HF_GGUF_URL], capture_output=True, text=True)
        if proc.returncode == 0:
            size = os.path.getsize(partial)
            if not expected or size == expected:
                os.replace(partial, dest)
                return dest, "verified"
            log(f"   size mismatch: got {size}, expected {expected}")
        else:
            log(f"   curl rc={proc.returncode}: {(proc.stderr or '')[-200:]}")
        time.sleep(3)

    log("⚠️ Primary HF download failed; trying the imatrix mirror (size-checked only)")
    mirror = (f"https://huggingface.co/{FALLBACK_REPO}/resolve/main/"
              f"Qwen3.8-27B-Uncensored.i1-{MODEL_QUANT}.gguf")
    if os.path.exists(partial):
        os.remove(partial)
    subprocess.run(["curl", "-fL", "--retry", "3", "-C", "-", "-o", partial, mirror], check=True)
    if os.path.getsize(partial) < 3 * 1024 * 1024 * 1024:
        raise RuntimeError(f"mirror download too small: {os.path.getsize(partial)} bytes")
    os.replace(partial, dest)
    return dest, "mirror"


def cache_dataset_is_ready(expected_bytes: int = 0) -> bool:
    """The published cache dataset already holds a full-size model?"""
    expected = expected_bytes or MODEL_BYTES.get(MODEL_QUANT, 0)
    try:
        req = urllib.request.Request(
            f"https://www.kaggle.com/api/v1/datasets/view/{CACHE_DATASET_SLUG}",
            headers={"Authorization": f"Bearer {KAGGLE_API_TOKEN}", "User-Agent": "CyberOrnith/1.0"},
        )
        with urllib.request.urlopen(req, timeout=20) as resp:
            data = json.loads(resp.read().decode("utf-8")) or {}
        total = int(data.get("totalBytesNullable") or data.get("totalBytes") or 0)
        return bool(total and (expected <= 0 or total >= expected))
    except Exception as exc:
        log(f"cache dataset check note: {exc}")
        return False


def publish_cache_dataset(gguf_path: str) -> None:
    """One-time: publish the verified GGUF as a private Kaggle dataset so future
    sessions mount it instantly instead of downloading 5.6 GB again."""
    try:
        os.makedirs(UPLOAD_DIR, exist_ok=True)
        target = os.path.join(UPLOAD_DIR, os.path.basename(gguf_path))
        if not _healthy_file(target, MODEL_BYTES.get(MODEL_QUANT, 0)):
            if os.path.exists(target):
                os.remove(target)
            log("📦 Seeding cache dataset upload dir (hardlink/copy of verified GGUF)...")
            try:
                os.link(gguf_path, target)
            except OSError:
                shutil.copy2(gguf_path, target)
        with open(os.path.join(UPLOAD_DIR, "dataset-metadata.json"), "w") as fh:
            json.dump(
                {
                    "title": CACHE_DATASET_TITLE,
                    "id": CACHE_DATASET_SLUG,
                    "licenses": [{"name": "other"}],
                },
                fh,
            )
        env = dict(os.environ)
        env["KAGGLE_USERNAME"] = KAGGLE_USERNAME
        env["KAGGLE_KEY"] = KAGGLE_API_TOKEN
        env["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
        env["KAGGLE_CONFIG_DIR"] = "/tmp/.kaggle"
        os.makedirs("/tmp/.kaggle", exist_ok=True)
        with open("/tmp/.kaggle/kaggle.json", "w") as fh:
            json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_API_TOKEN}, fh)
        os.chmod("/tmp/.kaggle/kaggle.json", 0o600)

        if shutil.which("kaggle") is None:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "--no-input", "kaggle"],
                check=False,
            )
        # Version the dataset when it already exists, otherwise create it.
        create = subprocess.run(
            ["kaggle", "datasets", "create", "-p", UPLOAD_DIR, "--dir-mode", "zip", "-q"],
            env=env, capture_output=True, text=True,
        )
        if create.returncode != 0:
            log(f"   create note: {create.stderr.strip()[:200]}")
            version = subprocess.run(
                ["kaggle", "datasets", "version", "-p", UPLOAD_DIR, "--dir-mode", "zip", "-q",
                 "-m", f"Cyber-Ornith {MODEL_QUANT} verified {time.strftime('%Y-%m-%d')}"],
                env=env, capture_output=True, text=True,
            )
            if version.returncode != 0:
                log(f"⚠️ Cache dataset publish failed: {version.stderr.strip()[:200]}")
                return
        log(f"✅ Cache dataset published: {CACHE_DATASET_SLUG}")
        notify_workspace("CACHE_DATASET_PUBLISHED", extra={"dataset": CACHE_DATASET_SLUG})
    except Exception as exc:
        log(f"⚠️ Cache dataset publish skipped: {exc}")


# --------------------- DEPENDENCY SETUP (skip when present) ---------------------
def ensure_python_deps() -> None:
    missing = []
    for module, package in (("fastapi", "fastapi"), ("uvicorn", "uvicorn"),
                            ("httpx", "httpx"), ("pydantic", "pydantic")):
        try:
            __import__(module)
        except Exception:
            missing.append(package)
    if missing:
        log(f"📦 Installing missing python packages: {missing}")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-input", *missing],
            check=False,
        )
    else:
        log("✅ Python deps already present — skipping pip install")


def ensure_ollama() -> str:
    binary = shutil.which("ollama")
    if binary:
        log(f"✅ Ollama already installed: {binary}")
        return binary
    log("🦙 Installing Ollama (not present in this fresh container)...")
    for tool in ("zstd", "pciutils", "lshw"):
        if shutil.which(tool) is None:
            subprocess.run("apt-get update -qq && apt-get install -y -qq zstd pciutils lshw",
                           shell=True, check=False)
            break
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    binary = shutil.which("ollama") or "/usr/local/bin/ollama"
    return binary


def start_ollama_daemon(ollama_bin: str) -> None:
    """Idempotent: bring the engine up as early as possible (during the download)."""
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags", timeout=2)
        log("✅ Ollama daemon already serving")
        return
    except Exception:
        pass
    os.makedirs(PERSIST_MODELS_DIR, exist_ok=True)
    env = os.environ.copy()
    env["OLLAMA_HOST"] = f"127.0.0.1:{OLLAMA_PORT}"
    env["OLLAMA_ORIGINS"] = "*"
    env["OLLAMA_KEEP_ALIVE"] = "-1"
    env["OLLAMA_MODELS"] = PERSIST_MODELS_DIR
    subprocess.Popen([ollama_bin, "serve"], env=env,
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags", timeout=2)
            log("✅ Ollama daemon up")
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError("ollama serve did not come up in 60s")


def ensure_ollama_serving() -> str:
    """Install (only if missing) *and* start the engine — runs during the download."""
    binary = ensure_ollama()
    start_ollama_daemon(binary)
    return binary


def ensure_cloudflared() -> str:
    for candidate in ("/tmp/cloudflared", "/usr/local/bin/cloudflared"):
        if os.path.exists(candidate):
            log(f"✅ cloudflared present: {candidate}")
            return candidate
    binary = "/tmp/cloudflared"
    log("🌐 Fetching cloudflared...")
    subprocess.run([
        "curl", "-fsSL",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-o", binary,
    ], check=True)
    os.chmod(binary, 0o755)
    return binary


def ollama_model_present() -> bool:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags", timeout=5) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        names = [m.get("name", "") for m in (data.get("models") or [])]
        return any(n.split(":")[0] == OLLAMA_MODEL_NAME.split(":")[0] for n in names)
    except Exception:
        return False


def register_model(ollama_bin: str, gguf_path: str) -> None:
    if ollama_model_present():
        log("✅ Model already registered in the ollama store — skipping create")
        return
    modelfile = "/tmp/Modelfile"
    with open(modelfile, "w") as fh:
        fh.write(f"FROM {gguf_path}\n")
        fh.write(f"PARAMETER num_ctx 8192\n")
        fh.write("PARAMETER temperature 0.6\n")
        fh.write('SYSTEM """' + DEFAULT_SYSTEM_PROMPT + '"""\n')
    log(f"🧱 Registering model from {gguf_path} (one-time copy into the local store)...")
    proc = subprocess.run([ollama_bin, "create", OLLAMA_MODEL_NAME, "-f", modelfile],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"ollama create failed: {proc.stderr.strip()[:300]}")
    log("✅ Model registered")


# --------------------- GATEWAY + TUNNEL ---------------------
def gateway_code() -> str:
    return '''
import json, time, os
import httpx
from fastapi import Depends, FastAPI, Header, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse

API_KEY = "''' + API_KEY + '''"
MODEL_ALIAS = "''' + MODEL_ALIAS + '''"
OLLAMA_MODEL = "''' + OLLAMA_MODEL_NAME + '''"
OLLAMA_URL = "http://127.0.0.1:''' + str(OLLAMA_PORT) + '''"
DEFAULT_SYSTEM_PROMPT = """''' + DEFAULT_SYSTEM_PROMPT + '''"""
START_TIME = time.time()
BOOT_STATE = json.load(open("/tmp/boot_state.json")) if os.path.exists("/tmp/boot_state.json") else {}
LOGS = "/tmp/cyber_ornith.log"

app = FastAPI(title="Cyber-Ornith API Server", version="4.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])


async def verify_api_key(authorization: str = Header(default=None), x_api_key: str = Header(default=None)):
    token = None
    if authorization and authorization.lower().startswith("bearer "):
        token = authorization[7:].strip()
    elif x_api_key:
        token = x_api_key.strip()
    if token != API_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing API Key.")
    return token


@app.get("/")
@app.get("/health")
async def health():
    """Liveness plus the honest model state.

    ``model_loaded`` is read from the engine itself (/api/ps), so the backend can
    keep saying "Loading Model" until the weights are really resident instead of
    advertising Ready while the first request would still queue behind a load.
    """
    model_loaded = False
    try:
        async with httpx.AsyncClient(timeout=4) as client:
            ps = await client.get(f"{OLLAMA_URL}/api/ps")
            if ps.status_code == 200:
                names = [str(m.get("name") or m.get("model") or "") for m in (ps.json().get("models") or [])]
                model_loaded = any(n.split(":")[0] == OLLAMA_MODEL.split(":")[0] for n in names)
    except Exception:
        model_loaded = bool(BOOT_STATE.get("warmup_ok"))
    return {
        "status": "online",
        "model": MODEL_ALIAS,
        "backend_model": OLLAMA_MODEL,
        "model_loaded": model_loaded,
        "gpu": BOOT_STATE.get("gpu", ""),
        "uptime_seconds": round(time.time() - START_TIME, 1),
        "boot": {k: BOOT_STATE.get(k) for k in ("cache_source", "boot_seconds", "download_skipped", "warmup_ok")},
        "endpoints": ["/v1/models", "/v1/chat/completions", "/chat", "/logs", "/health"],
        "auth_required": True,
    }


@app.get("/logs")
async def logs(dependencies=[Depends(verify_api_key)]):
    try:
        with open(LOGS, "r", errors="ignore") as fh:
            return {"lines": fh.read().splitlines()[-200:]}
    except Exception as exc:
        return {"lines": [], "error": str(exc)}


@app.get("/v1/models")
async def list_models(dependencies=[Depends(verify_api_key)]):
    return {"object": "list", "data": [{"id": MODEL_ALIAS, "object": "model",
            "created": int(START_TIME), "owned_by": "DuoNeural",
            "backing_gguf": OLLAMA_MODEL}]}


@app.post("/v1/chat/completions")
async def chat_completions(request: Request, dependencies=[Depends(verify_api_key)]):
    body = await request.json()
    messages = body.get("messages", [])
    if not any(m.get("role") == "system" for m in messages):
        messages = [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT}] + messages
    body["model"] = OLLAMA_MODEL
    body["messages"] = messages
    body.setdefault("keep_alive", -1)
    if body.get("stream"):
        async def event_generator():
            timeout = httpx.Timeout(connect=10.0, read=900.0, write=60.0, pool=10.0)
            async with httpx.AsyncClient(timeout=timeout) as client:
                async with client.stream("POST", f"{OLLAMA_URL}/v1/chat/completions", json=body) as resp:
                    async for chunk in resp.aiter_bytes():
                        yield chunk
        return StreamingResponse(event_generator(), media_type="text/event-stream")
    timeout = httpx.Timeout(connect=10.0, read=900.0, write=60.0, pool=10.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        resp = await client.post(f"{OLLAMA_URL}/v1/chat/completions", json=body)
        data = resp.json()
        if isinstance(data, dict):
            data["model"] = MODEL_ALIAS
        return JSONResponse(content=data, status_code=resp.status_code)


@app.post("/chat")
async def simple_chat(request: Request, dependencies=[Depends(verify_api_key)]):
    body = await request.json()
    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": body.get("system", DEFAULT_SYSTEM_PROMPT)},
            {"role": "user", "content": body.get("prompt") or body.get("message") or ""},
        ],
        "temperature": body.get("temperature", 0.6),
        "max_tokens": body.get("max_tokens", 1024),
        "stream": False,
    }
    timeout = httpx.Timeout(connect=10.0, read=900.0, write=60.0, pool=10.0)
    async with httpx.AsyncClient(timeout=timeout) as client:
        resp = await client.post(f"{OLLAMA_URL}/v1/chat/completions", json=payload)
        data = resp.json()
        try:
            reply = data["choices"][0]["message"]["content"]
        except Exception:
            reply = ""
        return {"model": MODEL_ALIAS, "response": reply}
'''


def write_gateway() -> str:
    path = "/tmp/gateway_server.py"
    with open(path, "w") as fh:
        fh.write(gateway_code())
    return path


def start_gateway() -> None:
    write_gateway()
    subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "gateway_server:app", "--host", "0.0.0.0",
         "--port", str(GATEWAY_PORT), "--log-level", "warning"],
        cwd="/tmp", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for _ in range(30):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=2)
            return
        except Exception:
            time.sleep(1)
    raise RuntimeError("Gateway did not come up on port 8000")


def launch_cloudflared(cloudflared_bin: str):
    cf_log_path = "/tmp/cloudflared.log"
    handle = open(cf_log_path, "w")
    proc = subprocess.Popen(
        [cloudflared_bin, "tunnel", "--url", f"http://127.0.0.1:{GATEWAY_PORT}", "--no-autoupdate"],
        stdout=handle, stderr=subprocess.STDOUT, text=True,
    )
    found = None
    deadline = time.time() + 45
    while time.time() < deadline:
        try:
            with open(cf_log_path, "r", errors="ignore") as fh:
                match = re.search(r"(https://[a-zA-Z0-9-]+\.trycloudflare\.com)", fh.read())
            if match:
                found = match.group(1)
                break
        except Exception:
            pass
        time.sleep(0.3)
    return proc, found


def main() -> None:
    started = time.time()
    boot_state = {"gpu": "", "cache_source": "", "download_skipped": False, "boot_seconds": 0}
    notify_workspace("CHECKING_ENVIRONMENT")

    try:
        gpu_out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]
        ).decode().strip()
        boot_state["gpu"] = gpu_out
        log(f"✅ GPU detected: {gpu_out}")
    except Exception:
        gpu_out = "CPU / No GPU attached"
        boot_state["gpu"] = gpu_out
        fail("GPU_UNAVAILABLE", "No NVIDIA GPU detected. Set Accelerator → GPU T4 x2 in Kaggle session options.")
        return

    try:
        urllib.request.urlopen("https://huggingface.co", timeout=8)
    except Exception as exc:
        fail("INTERNET_UNAVAILABLE", f"Kaggle internet is off ({exc}). Enable Internet in session options.")
        return

    # ---- installs run in the background while the model is fetched ----
    # These three steps are independent of the model transfer, so overlapping them
    # removes ~40-70 s from every cold start. The result arrives via futures.
    install_pool = concurrent.futures.ThreadPoolExecutor(max_workers=3)
    deps_future = install_pool.submit(ensure_python_deps)
    cloudflared_future = install_pool.submit(ensure_cloudflared)
    ollama_future = install_pool.submit(ensure_ollama_serving)
    notify_workspace("INSTALLING_DEPS", extra={"gpu": gpu_out, "note": "in parallel with the model transfer"})

    # ---- 1/2: find an existing persistent copy (no download) ----
    notify_workspace("CHECKING_CACHE", extra={"gpu": gpu_out})
    gguf = ""
    source = ""
    local_blob = find_local_store_model()
    if local_blob:
        gguf, source = local_blob, "persistent-store"

    if not gguf:
        mount = find_dataset_gguf()
        if mount:
            gguf, source = mount, "dataset"

    # ---- 3: download only when nothing valid exists ----
    if not gguf:
        notify_workspace("CACHE_MISS", extra={"gpu": gpu_out,
                                              "note": "no persistent copy mounted — downloading once"})
        notify_workspace("DOWNLOADING_MODEL", extra={"gpu": gpu_out, "note": "first run only"})
        dest = os.path.join(BLIND_DISK_DIR, GGUF_BASENAME.format(quant=MODEL_QUANT))
        # Speed-ordered sources: Hugging Face CDN measured ~60 s for the whole
        # 5.6 GB from inside Kaggle, while the dataset archive took ~5 min. The
        # dataset copy stays as the fallback so the GGUF is never fetched from HF
        # when a permanent Kaggle-side copy can serve it.
        try:
            gguf, how = download_model(dest)
        except Exception as exc:
            log(f"⚠️ Hugging Face source failed: {exc}")
            gguf, how = "", ""
        if not gguf:
            gguf = download_from_cache_dataset(dest)
            how = "kaggle-dataset" if gguf else ""
        if not gguf:
            fail("MODEL_CORRUPT", "no model source reachable — both Hugging Face and the Kaggle "
                                  "dataset copy failed; re-run the boot to retry")
            return
        expected_sha = MODEL_SHA256.get(MODEL_QUANT, "")
        if expected_sha and how in ("verified", "kaggle-dataset"):
            notify_workspace("VERIFYING_MODEL", extra={"gpu": gpu_out})
            actual = _sha256(gguf)
            if actual != expected_sha:
                # Retry once from a clean slate before declaring the model unusable.
                log(f"⚠️ sha256 mismatch ({actual[:12]}…) — re-downloading from scratch once")
                try:
                    os.remove(gguf)
                except OSError:
                    pass
                gguf, how = download_model(dest)
                if how == "verified" and _sha256(gguf) != expected_sha:
                    try:
                        os.remove(gguf)
                    except OSError:
                        pass
                    fail("MODEL_CORRUPT", "the model download failed its sha256 check twice; "
                                          "re-run the boot to retry (resume is safe)")
                    return
            _write_marker(gguf, expected_sha)
            log("✅ sha256 verified")
        else:
            log(f"⚠️ mirror source used ({how}) — size-checked only, sha256 not comparable")
        source = "download"
        boot_state["download_skipped"] = False
    else:
        boot_state["download_skipped"] = True
        log(f"⏭️  Download skipped — reusing {source} model")

    notify_workspace("CACHE_HIT" if boot_state["download_skipped"] else "MODEL_DOWNLOADED",
                     extra={"gpu": gpu_out, "source": source})

    # ---- collect the parallel installs (only the missing ones were touched) ----
    try:
        deps_future.result(timeout=420)
        cloudflared_bin = cloudflared_future.result(timeout=420)
        ollama_bin = ollama_future.result(timeout=420)
    except Exception as exc:
        fail("BOOT_FAILED", f"dependency install failed: {exc}")
        return
    finally:
        install_pool.shutdown(wait=False)
    log("✅ Parallel install stage complete")

    # ---- engine should already be serving (it was started during the download) ----
    notify_workspace("STARTING_OLLAMA", extra={"gpu": gpu_out})
    start_ollama_daemon(ollama_bin)   # no-op when already up

    # ---- register + load the model on the GPU ----
    notify_workspace("LOADING_MODEL", extra={"gpu": gpu_out})
    register_model(ollama_bin, gguf)

    boot_state["cache_source"] = source
    boot_state["boot_seconds"] = round(time.time() - started, 1)
    with open("/tmp/boot_state.json", "w") as fh:
        json.dump(boot_state, fh)
    notify_workspace("STARTING_GATEWAY", extra={"gpu": gpu_out})
    start_gateway()

    cf_proc, tunnel_url = launch_cloudflared(cloudflared_bin)
    if not tunnel_url:
        fail("TUNNEL_ERROR", "Could not obtain a Cloudflare Quick Tunnel URL")
        return
    notify_workspace("TUNNEL_ONLINE", tunnel_url=tunnel_url, extra={"gpu": gpu_out})

    notify_workspace("WARMING_GPU", tunnel_url=tunnel_url, extra={"gpu": gpu_out})
    warmup_ok = False
    for attempt in range(3):
        try:
            warm_req = urllib.request.Request(
                f"http://127.0.0.1:{GATEWAY_PORT}/v1/chat/completions",
                data=json.dumps({
                    "model": MODEL_ALIAS,
                    "messages": [{"role": "user", "content": "Ping. Reply with PONG."}],
                    "max_tokens": 24,
                }).encode("utf-8"),
                headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"},
            )
            urllib.request.urlopen(warm_req, timeout=300)
            warmup_ok = True
            break
        except Exception as exc:
            log(f"⚠️ Warmup attempt {attempt + 1} note: {exc}")
            time.sleep(5)

    boot_state["warmup_ok"] = warmup_ok
    boot_state["boot_seconds"] = round(time.time() - started, 1)
    with open("/tmp/boot_state.json", "w") as fh:
        json.dump(boot_state, fh)
    log(f"🚀 Boot complete in {boot_state['boot_seconds']}s (source={source}, download_skipped={boot_state['download_skipped']})")

    notify_workspace(
        "MODEL_READY_AND_WARMED" if warmup_ok else "MODEL_READY_COLD",
        tunnel_url=tunnel_url,
        extra={"gpu": gpu_out, "source": source, "boot_seconds": boot_state["boot_seconds"]},
    )

    # ---- one-time: make the verified model persistent for every future run ----
    if source == "download":
        if cache_dataset_is_ready():
            log("✅ Cache dataset already holds the model — skipping republish")
            notify_workspace("CACHE_DATASET_READY", extra={"dataset": CACHE_DATASET_SLUG, "gpu": gpu_out})
        else:
            publish_cache_dataset(gguf)

    log(f"⏳ Server loop active for up to {MAX_RUNTIME_SECONDS // 3600}h")
    loop_start = time.time()
    while time.time() - loop_start < MAX_RUNTIME_SECONDS:
        time.sleep(30)
        if cf_proc.poll() is not None:
            log("⚠️ cloudflared exited; restarting tunnel")
            cf_proc, new_url = launch_cloudflared(cloudflared_bin)
            if new_url:
                tunnel_url = new_url
                notify_workspace("MODEL_READY_AND_WARMED", tunnel_url=tunnel_url,
                                 extra={"gpu": gpu_out, "source": source, "tunnel_restart": True})
        if int(time.time() - loop_start) % 150 < 30:
            notify_workspace("HEARTBEAT_ONLINE", tunnel_url=tunnel_url,
                             extra={"gpu": gpu_out, "source": source})


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:  # never leave the dashboard stuck on "starting"
        import traceback
        traceback.print_exc()
        fail("BOOT_FAILED", f"{type(exc).__name__}: {exc}")
        raise
